In [0]:
from pyspark.sql.functions import col, count, when, round

RAW_FILE_PATH = "/Volumes/workspace/default/bankingdata/bank-full.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ";")
    .csv(RAW_FILE_PATH)
)

total_rows = df.count()
distinct_rows = df.dropDuplicates().count()
duplicate_rows = total_rows - distinct_rows

print(f"Total rows: {total_rows}")
print(f"Distinct rows: {distinct_rows}")
print(f"Duplicate rows: {duplicate_rows}")
print(f"Columns: {len(df.columns)}")

null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

display(null_counts)

target_distribution = (
    df.groupBy("y")
      .count()
      .withColumn("percentage", round(col("count") * 100 / total_rows, 2))
)

display(target_distribution)

quality_summary = spark.createDataFrame(
    [(total_rows, distinct_rows, duplicate_rows, len(df.columns))],
    ["total_rows", "distinct_rows", "duplicate_rows", "column_count"]
)

display(quality_summary)

Total rows: 45211
Distinct rows: 45211
Duplicate rows: 0
Columns: 17


age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


y,count,percentage
yes,5289,11.7
no,39922,88.3


total_rows,distinct_rows,duplicate_rows,column_count
45211,45211,0,17
